# Step 1: Setup prerequisites

In [ ]:
import os
import sys
from pymongo import MongoClient

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

In [ ]:
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI =  os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI)
# Check the connection to the server
mongodb_client.admin.command("ping")

In [ ]:
# Set the LLM provider and passkey provided by your workshop instructor
# NOTE: LLM_PROVIDER can be set to one of "aws"/ "microsoft" / "google"
LLM_PROVIDER = "aws"
PASSKEY = "replace-with-passkey"

In [ ]:
# Obtain API keys from our AI model proxy and set them as environment variables-- DO NOT CHANGE
set_env([LLM_PROVIDER,"voyageai"], PASSKEY)

In [ ]:
from anthropic import AnthropicBedrock
anthropic_client = AnthropicBedrock(aws_region="us-west-2")
model_id = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

In [ ]:
def get_completion(data):
    system = data.get("system", "")
    messages = data["messages"]
    tools = data.get("tools", [])
    output_config = data.get("output_config", {})

    response = anthropic_client.messages.create(
        model=model_id,
        max_tokens=8192,
        temperature=0,
        system=system,
        messages=messages,
        tools=tools,
        output_config = output_config
    )
    result = response.model_dump(exclude_none=True)
    return {
        "content": result["content"],
        "stop_reason": result["stop_reason"],
        "input_tokens": result["usage"]["input_tokens"],
        "output_tokens": result["usage"]["output_tokens"]
    }

In [ ]:
get_completion({"messages": [{"role": "user", "content": "Hi, who are you?"}]})

# Step 2: Setup memory collections

In [ ]:
import json
from utils import create_index, create_search_index, check_index_ready

### **Do not change the values assigned to the variables below**

In [ ]:
# Database name
DB_NAME = "mongodb_genai_devday_memory"
# Name of the semantic memory collection
SEMANTIC_COLLECTION_NAME = "semantic"
# Name of the episodic memory colleciton
EPISODIC_COLLECTION_NAME = "episodic"
# Name of the vector search index
VS_INDEX_NAME = "vector_index"

In [ ]:
# Connect to the `SEMANTIC_COLLECTION_NAME` collection.
semantic_collection = mongodb_client[DB_NAME][SEMANTIC_COLLECTION_NAME]
# Connect to the `EPISODIC_COLLECTION_NAME` collection.
episodic_collection = mongodb_client[DB_NAME][EPISODIC_COLLECTION_NAME]

In [ ]:
# Seed episodic memories collection
with open(f"../data/memories.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {EPISODIC_COLLECTION_NAME} collection.")
episodic_collection.delete_many({})
episodic_collection.insert_many(data)
print(
    f"{episodic_collection.count_documents({})} documents ingested into the {EPISODIC_COLLECTION_NAME} collection."
)

In [ ]:
# Create vector index definition for the episodic memories collection:
# path: Path to the embeddings field
# numDimensions: Number of embedding dimensions- depends on the embedding model used
# similarity: Similarity metric. One of cosine, euclidean, dotProduct.
model = {
    "name": VS_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

In [ ]:
# Use the `create_search_index` function from the `utils` module to create a vector search index with the above definition for the `episodic_collection` collection
create_search_index(episodic_collection, VS_INDEX_NAME, model)

In [ ]:
# Use the `check_index_ready` function from the `utils` module to verify that the index was created and is in READY status before proceeding
check_index_ready(episodic_collection, VS_INDEX_NAME)

In [ ]:
create_index(semantic_collection, "timestamp", "ttl_index", expireAfterSeconds=31536000)

# Step 3: Define memory management tools

In [ ]:
from datetime import datetime
import voyageai

In [ ]:
voyage_client = voyageai.Client()

In [ ]:
def get_embedding(text: str, input_type: str) -> list[float]:
    """
    Get embeddings for an input text

    Args:
        text (str): Text to embed
        input_type (str): One of "document" or "query"

    Returns:
        list[float]: Embedding of the query string
    """
    result = voyage_client.embed(texts=[text], model="voyage-4", input_type=input_type)
    return result.embeddings[0]

In [ ]:
def save_user_memories(user_id: str, memories: list[str]) -> str:
    """
    Save user facts and preferences

    Args:
        user_id (str): User ID
        memories (list[str]): Facts and preferences to save

    Returns:
        str: Save message
    """
    docs = [{"user_id": user_id, "content": m, "timestamp": datetime.utcnow()} for m in memories]
    semantic_collection.insert_many(docs)
    return f"Saved {len(memories)} memories"

In [ ]:
def get_user_memories(user_id: str) -> str:
    """
    Retrieve memories for a particular user

    Args:
        user_id (str): User ID

    Returns:
        str: Retrieved memories formatted as a string
    """
    memories = list(semantic_collection.find(
        {"user_id": user_id},
        {"_id": 0, "content": 1, "timestamp": 1}
    ).sort("timestamp", -1))
    
    if not memories:
        return f"No memories found for user {user_id}"
    
    lines = [f"- {m['timestamp']} {m['content']}" for m in memories]
    return f"Memories for {user_id}:\n" + "\n".join(lines)

In [ ]:
NOTES_FILE = "./NOTES.md"

In [ ]:
def write_notes(notes: list[str]) -> str:
    """
    Write notes to a local file

    Args:
        notes (list[str]): Notes to write to file

    Returns:
        str: Note recorded message
    """
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    with open(NOTES_FILE, "a") as f:
        for note in notes:
            f.write(f"- [{timestamp}] {note}\n")
    return f"Recorded {len(notes)} notes"

In [ ]:
def save_episode() -> str:
    """
    Generate an episodic memory from notes and save to MongoDB
    
    Returns:
        str: Episode save acknowledgment
    """
    # Read notes from file
    try:
        with open(NOTES_FILE, "r") as f:
            notes_content = f.read().strip()
    except FileNotFoundError:
        return "No notes found to create episode from."
    
    if not notes_content:
        return "Notes file is empty."
    
    # Use LLM to generate episode title and description
    prompt = f"""Based on these session notes, create a concise summary of what was accomplished, the approach taken, and key insights gained.
    Also, provide a brief, descriptive title for the episode.
    Notes:{notes_content}
    """

    response = get_completion(
        {
            "messages": [{"role": "user", "content": prompt}],
            "output_config": {
                "format": {
                    "type": "json_schema",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string", "description": "Brief, descriptive title (5-10 words)"},
                            "description": {"type": "string", "description": "A summary of what was accomplished, the approach taken, and key insights (max 2 paragraphs)"},
                        },
                        "required": ["title", "description"],
                        "additionalProperties": False,
                    },
                }
            }
        }
    )
    
    # Parse LLM response
    episode = json.loads(response["content"][0]["text"])
        
    # Generate embedding and save
    embedding = get_embedding(episode["description"], "document")
    episodic_collection.insert_one({
        "title": episode["title"],
        "description": episode["description"],
        "timestamp": datetime.utcnow(),
        "embedding": embedding
    })
    
    # Clear the notes file after saving
    open(NOTES_FILE, "w").close()
    
    return f"Episode saved: '{episode['title']}' - Notes cleared."

In [ ]:
def get_episodes(query: str) -> str:
    """
    Retrieve memories of how you solved previous problems, using semantic search

    Args:
        note (str): The 

    Returns:
        str: Relevant episodes formatted as strings
    """
    query_embedding = get_embedding(query, "query")
    
    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": 5
            }
        },
        {"$project": {"_id": 0, "title": 1, "description": 1, "score": {"$meta": "vectorSearchScore"}}}
    ]
    
    results = list(episodic_collection.aggregate(pipeline))
    
    if not results:
        return "No relevant episodes found."
    
    output = "Relevant past episodes:\n"
    for i, ep in enumerate(results, 1):
        output += f"{i}. **{ep['title']}** (relevance: {ep['score']:.2f})\n"
        output += f"{ep['description']}\n\n"
    return output

# Step 4: Define tool schemas

In [ ]:
memory_tools = [
    {
        "name": "save_user_memories",
        "description": "Save user preferences or facts as memories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "array", 
                    "items": {"type": "string"},
                    "description": "List of memories to save"
                },
            },
            "required": ["content"]
        }
    },
    {
        "name": "get_user_memories",
        "description": "Retrieve stored memories for a user. Use at conversation start to get the user's profile.",
        "input_schema": {                    
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "write_notes",
        "description": "Write notes about the current task to a local file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "notes": {
                    "type": "array", 
                    "items": {"type": "string"},
                    "description": "List of notes to write to file."
                },
            },
            "required": ["notes"]
        }
    },
    {
        "name": "save_episode",
        "description": "Create an episodic memory from accumulated notes.",
        "input_schema": {                    # Add this
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "get_episodes",
        "description": "Search past episodic memories by semantic similarity.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Description of the problem or topic to search"
                },
            },
            "required": ["query"]
        }
    }
]

# Step 5: Define tool execution function

In [ ]:
def execute_tool(tool_name: dict, tool_args: dict, session: dict) -> str:
    """
    Coordinate tool execution

    Args:
        tool_name (str): Tool name
        tool_args (dict): Arguments for the tool call
        session (dict): Session information

    Returns:
        str: Tool output formatted as string
    """
    if tool_name == "save_user_memories":
        return save_user_memories(session["user_id"], tool_args["content"])
    elif tool_name == "get_user_memories":
        return get_user_memories(session["user_id"])
    elif tool_name == "write_notes":
        return write_notes(tool_args["notes"])
    elif tool_name == "save_episode":
        return save_episode()
    elif tool_name == "get_episodes":
        return get_episodes(tool_args["query"])
    else:
        return f"Unknown tool: {tool_name}"

# Step 6: Create a memory-augmented agent

In [ ]:
import uuid

### Step 6a: Create a session management function

In [ ]:
def create_session(user_id: str, max_tokens: int=200000) -> dict:
    """
    Create a session

    Args:
        user_id (str): Unique user identifier
        max_tokens (int): Context window limit of the LLM used

    Returns:
        dict: Session object
    """
    return {
        "session_id": str(uuid.uuid4())[:8],
        "user_id": user_id,
        "messages": [],
        "input_tokens": 0,
        "output_tokens": 0,
        "max_tokens": max_tokens
    }

### Step 6b: Define a dynamic system prompt for the agent

In [ ]:
def get_context_usage(session: dict) -> float:
    """
    Calculate context window usage as a percentage

    Args:
        session (dict): Session object

    Returns:
        float: % of the context window used
    """
    total_tokens = session["input_tokens"] + session["output_tokens"]
    return (total_tokens / session["max_tokens"]) * 100

In [ ]:
def get_system_prompt(session: dict) -> str:
    """
    Build system prompt with context usage status

    Args:
        session (dict): Session object

    Returns:
        str: Agent's system prompt
    """
    usage_pct = get_context_usage(session)
    prompt = f"""You are a helpful coding assistant with memory capabilities.

    ## CURRENT SESSION STATUS
    - Context window usage: {usage_pct:.1f}%

    ## GENERAL INSTRUCTIONS
    - Ask additional questions if user requests are unclear or broad.
    - Keep responses concise and focused. Avoid lengthy code examples unless specifically requested.

    ## MEMORY PROTOCOL
    **At conversation start:**
    - Use `get_user_memories` to retrieve user preferences and facts.
    - Prioritize RECENT information if memories conflict.

    **When to search episodes (`get_episodes`):**
    - ANY technical or coding question
    - Episodes are YOUR internal knowledge from past sessions. Use them SILENTLY to inform your approach.

    **During the conversation:**
    - Use `write_notes` to record progress, decisions, and insights.
    - Keep EACH NOTE to 2-3 SENTENCES.
    - Write notes often— especially after code explanations or key decisions.

    **When you learn something important about the user:**
    - Use `save_user_memories` to save facts and preferences such as preferred language, frameworks, coding style, and experience level.
    - Keep EACH MEMORY to 1 SENTENCE.

    **When to save an episode (`save_episode`):**
    - A logical unit of work is COMPLETE — you explained a concept, provided a solution, or answered a question fully.                                                                                                                                                                                                    
    - The user has ACKNOWLEDGED the solution (verbally or by moving on).
    - You are about to switch to a DIFFERENT problem domain.
    - Save episodes to capture COMPLETE insights, not partial work-in-progress.
    - Episode summary should NOT EXCEED 2 PARAGRAPHS.
    - Definitely save an episode when the context window usage exceeds 70%.

    ASSUME INTERRUPTION: Your context window might be reset at any moment, so you risk losing any progress that is not recorded in your notes.
    """
    
    return prompt

### Step 6c: Define the agent loop

In [ ]:
def chat(user_input: str, session: dict) -> str:
    """
    The agent execution loop

    Args:
        user_input (str): The user's question
        session (dict): Session object

    Returns:
        str: Agent's response
    """
    # Add user message
    print("===== Human message =====")
    print(user_input)
    session["messages"].append({"role": "user", "content": user_input})
    
    while True:
        # Build dynamic system prompt with current context status
        system_prompt = get_system_prompt(session)
        
        # Call LLM
        response = get_completion(
            {
                "system": system_prompt,
                "messages": session["messages"],
                "tools": memory_tools
            }
        )
        
        # Update token counts
        session["input_tokens"] += response["input_tokens"]
        session["output_tokens"] += response["output_tokens"]
        
        # Add assistant response to history
        content = response["content"]
        print("===== AI message =====")
        text_parts = [block["text"] for block in content if block.get("type") == "text"]
        if text_parts:
            print("\n".join(text_parts))
        session["messages"].append({"role": "assistant", "content": content})
        
        # Final response - return to user
        if response["stop_reason"] == "end_turn":
            break
                
        # Handle tool calls
        elif response["stop_reason"] == "tool_use":
            tool_results = []
            for block in content:
                if block.get("type") == "tool_use":
                    tool_name = block["name"]
                    tool_args = block["input"]
                    print("===== Tool Call =====")
                    print(f"Calling {tool_name} with args: {tool_args}")
                    
                    # Execute tool
                    result = execute_tool(tool_name, tool_args, session)
                    print(f"===== Tool Outcome: {tool_name} =====")
                    print(result)
                    
                    # Collect tool results
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block["id"],
                        "content": result
                    })
            # Add ALL tool results in a single message
            session["messages"].append({
                "role": "user",
                "content": tool_results
            })

# Step 7: Interact with the agent

In [ ]:
# Create a new session
session = create_session(user_id="mdb_user")
print(f"Started session: {session['session_id']}")

In [ ]:
# Test 1: Ask a coding question
# The agent should take notes and potentially search for relevant past episodes
response = chat("I'm building an AI-powered shopping assistant. Let's start by building the chat interface.", session)

In [ ]:
response = chat("I prefer FastAPI for the backend, Claude as the LLM and MongoDB to store chat history.", session)